# Vectorización: La Bolsa de Palabras (Bag of Words)

---

**Objetivo:** Comprender y aplicar la técnica de Bag of Words (BoW) para transformar textos en representaciones numéricas que una computadora pueda procesar.

**Resultados de aprendizaje:** Al finalizar este cuaderno, vas a poder:
1. Explicar con tus palabras la intuición detrás de la técnica Bag of Words.
2. Construir manualmente una matriz de conteos a partir de un corpus pequeño.
3. Automatizar ese proceso usando `CountVectorizer` de `scikit-learn`.
4. Aplicar BoW a un caso real (detección de spam) e interpretar los resultados.
5. Usar BoW para un análisis básico de sentimiento en reseñas de productos.

**Relación con cuadernos anteriores:** En encuentros previos aprendimos a adquirir textos y a limpiarlos (tokenización, normalización). Ahora damos el paso clave: convertir esas palabras en números para que un algoritmo pueda operar con ellas.

## Sección 1: Concepto — La analogía de la bolsa

Imaginá que tenés una bolsa vacía y un texto impreso en papel. Para "resumir" de qué trata ese texto sin leerlo en orden, podés hacer lo siguiente:

1. **Tomás el texto** → "El perro muerde al perro".
2. **Cortás cada palabra** (tokens) → `["El", "perro", "muerde", "al", "perro"]`.
3. **Las metés en la bolsa** y la agitás (se pierde el orden original).
4. **Contás cuántas hay de cada una** → obtenés un perfil numérico del texto.

### Resultado: un inventario de frecuencias

| Palabra | Cantidad |
|---------|----------|
| El      | 1        |
| perro   | 2        |
| muerde  | 1        |
| al      | 1        |

**Eso es Bag of Words.** Convertimos lenguaje humano en una lista de números que representan cuántas veces aparece cada palabra.

## Sección 2: Ejercicio manual

Antes de escribir código, vamos a hacer este proceso a mano para fijar la intuición.

### Nuestro corpus de ejemplo
1. "El perro ladra fuerte"
2. "El gato come pescado"
3. "El perro come carne"

### Paso 1 — Crear el vocabulario
El vocabulario es el conjunto de **todas las palabras únicas** del corpus:

*Vocabulario:* **carne, come, el, fuerte, gato, ladra, perro, pescado** (8 términos, ordenados alfabéticamente).

### Paso 2 — Construir la matriz de conteos
Cada fila es un documento, cada columna una palabra del vocabulario.

| Documento | carne | come | el | fuerte | gato | ladra | perro | pescado |
|-----------|-------|------|----|--------|------|-------|-------|---------|
| Doc 1     | 0     | 0    | 1  | 1      | 0    | 1     | 1     | 0       |
| Doc 2     | 0     | 1    | 1  | 0      | 1    | 0     | 0     | 1       |
| Doc 3     | 1     | 1    | 1  | 0      | 0    | 0     | 1     | 0       |

Observá que el vocabulario está en orden alfabético. `CountVectorizer` de Python hace exactamente lo mismo.

## Sección 3: Automatización con Python

Ahora que entendés el proceso manual, vamos a automatizarlo. En proyectos reales el vocabulario tiene miles de palabras y el corpus miles de documentos, así que hacerlo a mano sería imposible.

Vamos a usar `CountVectorizer` de la librería `scikit-learn`. Esta herramienta hace exactamente los dos pasos que hicimos a mano:
- **`fit`**: recorre todos los documentos y arma el vocabulario (las columnas de nuestra tabla).
- **`transform`**: cuenta cuántas veces aparece cada palabra del vocabulario en cada documento.

Cuando usamos `fit_transform`, ejecutamos ambos pasos en una sola llamada.

In [17]:
# --- Paso 1: Preparamos el entorno ---

# CountVectorizer: herramienta de scikit-learn que construye
# el vocabulario y cuenta las frecuencias automáticamente.
from sklearn.feature_extraction.text import CountVectorizer

# pandas: librería para mostrar los resultados en tablas legibles.
import pandas as pd

# --- Paso 2: Definimos el corpus ---
# Usamos los mismos 3 textos del ejercicio manual
# para poder comparar los resultados.
corpus_ejemplo = [
    "El perro ladra fuerte",
    "El gato come pescado",
    "El perro come carne"
]

print("Nuestro corpus de ejemplo:")
for i, documento in enumerate(corpus_ejemplo):
    print(f"  Documento {i+1}: {documento}")

Nuestro corpus de ejemplo:
  Documento 1: El perro ladra fuerte
  Documento 2: El gato come pescado
  Documento 3: El perro come carne


A continuación, le pasamos nuestro corpus a `CountVectorizer`. Esperamos obtener:
- Un **vocabulario** con las 8 palabras únicas (en minúsculas y orden alfabético).
- Una **matriz** de 3 filas × 8 columnas con los conteos.

In [18]:
# --- Paso 3: Creamos el contador y vectorizamos ---

# Inicializamos CountVectorizer (sin parámetros usa la configuración por defecto:
# tokeniza por espacios, pasa a minúsculas, ignora signos de puntuación).
contador = CountVectorizer()

# fit_transform hace dos cosas en una:
#   1. fit  → aprende el vocabulario del corpus
#   2. transform → cuenta las ocurrencias de cada palabra en cada documento
# El resultado es una "matriz esparsa" (formato eficiente donde sólo se
# guardan los valores distintos de cero).
matriz_conteo = contador.fit_transform(corpus_ejemplo)

# Recuperamos el vocabulario que identificó el algoritmo
vocabulario = contador.get_feature_names_out()

print("Vocabulario identificado (en orden alfabético):")
print(list(vocabulario))
print(f"\nCantidad de términos únicos: {len(vocabulario)}")

Vocabulario identificado (en orden alfabético):
['carne', 'come', 'el', 'fuerte', 'gato', 'ladra', 'perro', 'pescado']

Cantidad de términos únicos: 8


Fijate que `CountVectorizer` convirtió todo a minúsculas automáticamente. Por eso "El" aparece como "el". Ahora veamos la matriz completa en forma de tabla.

In [19]:
# --- Paso 4: Visualizamos la matriz como tabla ---

# Convertimos la matriz esparsa a una tabla legible (DataFrame de pandas).
# .toarray() transforma la representación eficiente en una matriz completa.
tabla_frecuencias = pd.DataFrame(
    matriz_conteo.toarray(),
    columns=vocabulario,       # las palabras del vocabulario como encabezado
    index=["Doc 1", "Doc 2", "Doc 3"]  # nombres para cada fila
)

print("Tabla de frecuencias (Bag of Words):")
tabla_frecuencias

Tabla de frecuencias (Bag of Words):


,carne,come,el,fuerte,gato,ladra,perro,pescado
Doc 1,0,0,1,1,0,1,1,0
Doc 2,0,1,1,0,1,0,0,1
Doc 3,1,1,1,0,0,0,1,0


## Sección 4: Reflexión guiada

Mirá con detenimiento la tabla anterior. Cada fila es ahora una "huella digital" numérica de cada texto.

1. **¿Qué documentos son más parecidos entre sí?** Compará las filas: ¿cuáles comparten más columnas con valores iguales?
2. **¿Qué palabra aparece en los tres documentos?** ¿Creés que esa palabra ayuda a distinguir de qué trata cada texto?
3. **¿Qué términos son exclusivos de un solo documento?** Esos son los que mejor definen el contenido particular de cada texto.

### Conclusiones clave
- **Doc 1 y Doc 3** comparten "el" y "perro" → el algoritmo "ve" que tienen un tema en común.
- La palabra **"el"** aparece en todos los documentos. No aporta información distintiva. En el futuro vamos a aprender a filtrar este tipo de palabras (se llaman **stopwords**).

## Sección 5: Ejemplo práctico — Detección de spam

¿Cómo usa Gmail esta idea para filtrar correo basura? La lógica es simple:
- Vectoriza los correos con BoW.
- Observa qué palabras son frecuentes en correos spam ("oferta", "gratis", "urgente").
- Observa qué palabras son frecuentes en correos legítimos ("reunión", "informe", "mamá").
- Clasifica los nuevos correos según la similitud de sus vectores con cada grupo.

Vamos a reproducir esta idea con un corpus pequeño de 5 correos.

In [20]:
# --- Paso 1: Definimos una pequeña base de emails ---

emails = [
    "Reunión mañana a las 9am en la oficina",       # Legítimo
    "OFERTA ESPECIAL! COMPRE AHORA! DINERO FÁCIL!", # Spam
    "Hola mamá, ¿cómo estás? Te extraño mucho",     # Legítimo
    "¡¡¡GANE MILLONES!!! CLICK AQUÍ URGENTE!!!",    # Spam
    "Adjunto el informe que me pediste ayer"         # Legítimo
]

# Etiquetas reales (las sabemos nosotros, no el algoritmo)
etiquetas = ["Legítimo", "Spam", "Legítimo", "Spam", "Legítimo"]

print("Emails de prueba:")
for i, email in enumerate(emails):
    print(f"  {i+1}. [{etiquetas[i]}] {email}")

Emails de prueba:
  1. [Legítimo] Reunión mañana a las 9am en la oficina
  2. [Spam] OFERTA ESPECIAL! COMPRE AHORA! DINERO FÁCIL!
  3. [Legítimo] Hola mamá, ¿cómo estás? Te extraño mucho
  4. [Spam] ¡¡¡GANE MILLONES!!! CLICK AQUÍ URGENTE!!!
  5. [Legítimo] Adjunto el informe que me pediste ayer


Ahora vectorizamos estos correos y filtramos solo los términos más reveladores para ver si podemos distinguir spam de legítimo con BoW.

In [21]:
# --- Paso 2: Vectorizamos los emails ---

# Creamos un nuevo contador (independiente del anterior)
contador_emails = CountVectorizer()
matriz_emails = contador_emails.fit_transform(emails)

vocabulario_emails = contador_emails.get_feature_names_out()

# Armamos la tabla completa
tabla_emails = pd.DataFrame(
    matriz_emails.toarray(),
    columns=vocabulario_emails,
    index=[f"Email {i+1} ({etiquetas[i]})" for i in range(len(emails))]
)

# La tabla completa tiene muchas columnas; seleccionamos solo
# los términos más representativos para poder analizarla.
terminos_clave = ["oferta", "especial", "dinero", "gane",
                  "millones", "urgente", "reunión", "mamá", "informe"]
columnas_visibles = [col for col in terminos_clave if col in tabla_emails.columns]

print(f"Vocabulario total: {len(vocabulario_emails)} términos únicos.")
print(f"Mostramos solo {len(columnas_visibles)} términos clave:\n")
tabla_emails[columnas_visibles]

Vocabulario total: 32 términos únicos.
Mostramos solo 9 términos clave:



,oferta,especial,dinero,gane,millones,urgente,reunión,mamá,informe
Email 1 (Legítimo),0,0,0,0,0,0,1,0,0
Email 2 (Spam),1,1,1,0,0,0,0,0,0
Email 3 (Legítimo),0,0,0,0,0,0,0,1,0
Email 4 (Spam),0,0,0,1,1,1,0,0,0
Email 5 (Legítimo),0,0,0,0,0,0,0,0,1


### ¿Qué observamos?

- Los **emails de spam** (2 y 4) tienen valores altos en columnas como "oferta", "dinero", "gane", "millones", "urgente".
- Los **emails legítimos** (1, 3 y 5) tienen valores en columnas como "reunión", "mamá", "informe".
- No hay solapamiento: cada grupo usa un vocabulario claramente distinto.

Un algoritmo de clasificación (como Naive Bayes, que veremos más adelante) puede aprender estos patrones automáticamente a partir de la matriz BoW.

## Sección 6: Limitaciones y próximos pasos

Bag of Words es una técnica potente y simple, pero tiene puntos ciegos:

1. **Se pierde el orden:** "No me gusta nada" y "Me gusta, no nada" producen el mismo vector aunque signifiquen cosas opuestas.
2. **El contexto es invisible:** "banco" (de sentarse) y "banco" (financiero) se cuentan como la misma palabra.
3. **Vocabularios gigantes:** Con millones de documentos, la matriz crece enormemente y se vuelve costosa de procesar.

En los próximos encuentros vamos a ver:
- **TF-IDF**: una mejora que pondera la importancia real de cada palabra.
- **Word Embeddings**: representaciones que capturan el significado profundo de las palabras.

## Actividad de cierre: Análisis de sentimiento con BoW

Para cerrar, vamos a aplicar lo que aprendimos a un problema diferente: **clasificar reseñas de productos como positivas o negativas** a partir de sus palabras.

Observá los siguientes comentarios de una tienda online. Vamos a vectorizarlos y ver si los términos nos permiten distinguir la polaridad.

In [22]:
# --- Reseñas de una tienda online ---

resenias = [
    "Excelente producto, muy buena calidad, lo recomiendo",         # Positiva
    "Terrible, no funciona, quiero mi dinero de vuelta",           # Negativa
    "Perfecto, justo lo que necesitaba, cinco estrellas",          # Positiva
    "Pésimo servicio, nunca llegó, muy decepcionado",              # Negativa
    "Increíble calidad, superó mis expectativas completamente"     # Positiva
]

polaridad = ["Positiva", "Negativa", "Positiva", "Negativa", "Positiva"]

print("Reseñas a analizar:")
for i, resenia in enumerate(resenias):
    print(f"  {i+1}. [{polaridad[i]}] {resenia}")

Reseñas a analizar:
  1. [Positiva] Excelente producto, muy buena calidad, lo recomiendo
  2. [Negativa] Terrible, no funciona, quiero mi dinero de vuelta
  3. [Positiva] Perfecto, justo lo que necesitaba, cinco estrellas
  4. [Negativa] Pésimo servicio, nunca llegó, muy decepcionado
  5. [Positiva] Increíble calidad, superó mis expectativas completamente


Ahora vectorizamos las reseñas y seleccionamos los términos que mejor distinguen la opinión positiva de la negativa.

In [23]:
# --- Vectorizamos las reseñas ---

contador_resenias = CountVectorizer()
matriz_resenias = contador_resenias.fit_transform(resenias)
vocabulario_resenias = contador_resenias.get_feature_names_out()

tabla_resenias = pd.DataFrame(
    matriz_resenias.toarray(),
    columns=vocabulario_resenias,
    index=[f"Reseña {i+1} ({polaridad[i]})" for i in range(len(resenias))]
)

# Seleccionamos términos indicadores de cada polaridad
terminos_positivos = ["excelente", "buena", "recomiendo", "perfecto",
                      "increíble", "cinco", "estrellas", "superó"]
terminos_negativos = ["terrible", "pésimo", "nunca", "decepcionado",
                      "vuelta", "funciona"]

# Filtramos solo los que existen en el vocabulario
columnas_positivas = [t for t in terminos_positivos if t in vocabulario_resenias]
columnas_negativas = [t for t in terminos_negativos if t in vocabulario_resenias]
columnas_sentimiento = columnas_positivas + columnas_negativas

print(f"Vocabulario total: {len(vocabulario_resenias)} términos.")
print(f"Mostramos {len(columnas_sentimiento)} términos clave:\n")
tabla_resenias[columnas_sentimiento]

Vocabulario total: 31 términos.
Mostramos 14 términos clave:



,excelente,buena,recomiendo,perfecto,increíble,cinco,estrellas,superó,terrible,pésimo,nunca,decepcionado,vuelta,funciona
Reseña 1 (Positiva),1,1,1,0,0,0,0,0,0,0,0,0,0,0
Reseña 2 (Negativa),0,0,0,0,0,0,0,0,1,0,0,0,1,1
Reseña 3 (Positiva),0,0,0,1,0,1,1,0,0,0,0,0,0,0
Reseña 4 (Negativa),0,0,0,0,0,0,0,0,0,1,1,1,0,0
Reseña 5 (Positiva),0,0,0,0,1,0,0,1,0,0,0,0,0,0


### ¿Qué observamos?

- Las **reseñas positivas** (1, 3 y 5) concentran sus valores en las columnas de la izquierda ("excelente", "buena", "recomiendo", etc.).
- Las **reseñas negativas** (2 y 4) concentran sus valores en las columnas de la derecha ("terrible", "pésimo", "nunca", etc.).

Este patrón es exactamente el que un clasificador automático podría aprender: **la presencia o ausencia de ciertas palabras es un indicador fuerte de la polaridad del texto.**

---

### Cierre

En este cuaderno vimos cómo Bag of Words convierte texto en números que un algoritmo puede procesar. Aplicamos esta técnica a dos casos reales:
- **Detección de spam** en correos electrónicos.
- **Análisis de sentimiento** en reseñas de productos.

En ambos casos, la simple frecuencia de palabras fue suficiente para distinguir categorías. Pero también vimos las limitaciones: BoW ignora el orden y el contexto de las palabras.

**En el próximo cuaderno** vamos a conocer **TF-IDF**, una evolución de BoW que no solo cuenta palabras, sino que pondera su importancia relativa dentro del corpus.